##**Part 3 — Machine Learning: predict no-shows and choose the model honestly**













##**Task 3.1 — Features, split and baseline**


In [1]:
# Import libraries

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
)

import pandas as pd

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# 1. Load the clean appointments dataset
original_clean_df = pd.read_csv('appointments_clean.csv')

# 2. Map reminder_sent Yes/No to 1/0
original_clean_df['reminder_sent_mapped'] = original_clean_df['reminder_sent'].str.strip().str.lower().map({'yes': 1, 'no': 0})

# 3. Select only the specified features
feature_cols = ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent_mapped', 'booking_channel', 'visit_type']
X_raw = original_clean_df[feature_cols].copy()

X_raw = X_raw.rename(columns={'reminder_sent_mapped': 'reminder_sent'})

# One-hot encode booking_channel and visit_type
X = pd.get_dummies(X_raw, columns=['booking_channel', 'visit_type'])

# Target variable
y = original_clean_df['no_show']

# 4. Split the dataset (test_size=0.2, random_state=42, no shuffling/sorting beforehand to keep row order)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Baseline: Always predict "Attended" (status 0)
baseline_clf = DummyClassifier(strategy="most_frequent")
baseline_clf.fit(X_train, y_train)
y_pred = baseline_clf.predict(X_test)

# 6. Report the required results
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Number of no-shows in test set: {y_test.sum()}")
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Training set size: 792
Test set size: 198
Number of no-shows in test set: 49
Baseline Accuracy: 0.7525


#**Task 3.2 — Logistic regression**


In [4]:

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)

# 1. Scale features
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 2. Fit Logistic Regression
logreg_clf = LogisticRegression(max_iter=1000)
logreg_clf.fit(X_train_s, y_train)

# 3. Predict and evaluate
y_pred = logreg_clf.predict(X_test_s)

# Predicted probabilities for ROC-AUC
y_prob = logreg_clf.predict_proba(X_test_s)[:, 1]

print("=== Model Evaluation on Test Set ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# 4. Display feature coefficients
print("\n=== Feature Coefficients ===")

features = X_train.columns
coefs = logreg_clf.coef_[0]

for feat, co in zip(features, coefs):
    print(f"{feat}: {co:.4f}")

# 5. Reminder Sent odds-ratio analysis
idx_reminder = list(features).index('reminder_sent')

coef_reminder = coefs[idx_reminder]
std_reminder = scaler.scale_[idx_reminder]

# Calculate odds ratios
odds_ratio_scaled = np.exp(coef_reminder)

unscaled_coefficient = coef_reminder / std_reminder
odds_ratio_unscaled = np.exp(unscaled_coefficient)

print("\n=== Reminder Sent Analysis ===")
print(f"Scaled coefficient: {coef_reminder:.4f}")
print(f"Scaled odds ratio: {odds_ratio_scaled:.4f}")
print(f"Unscaled coefficient: {unscaled_coefficient:.4f}")
print(f"Unscaled odds ratio: {odds_ratio_unscaled:.4f}")

# 6. Explanation and conclusion
print("\n=== Conclusions ===")

print(
    "Why do the odds ratios differ?\n"
    "The scaled coefficient represents the change in log-odds "
    "for a one-standard-deviation increase in reminder_sent.\n"
    "The unscaled coefficient represents the change in log-odds "
    "when reminder_sent changes from 0 (No) to 1 (Yes)."
)

print("=" * 70)

print("\nConclusion:")
print(
    f"Holding other features constant, sending a reminder is "
    f"associated with an odds ratio of {odds_ratio_unscaled:.4f} "
    f"for no-show."
)

print(
    "An odds ratio below 1 indicates lower odds of no-show when "
    "a reminder is sent. This is an association, not proof of causation."
)


=== Model Evaluation on Test Set ===
Accuracy: 0.7879
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

=== Feature Coefficients ===
age: -0.2539
distance_km: 0.2534
fee_inr: 0.4528
lead_days: 0.6145
prior_no_shows: 0.6496
reminder_sent: -0.5807
booking_channel_App: -0.0468
booking_channel_Phone: 0.2159
booking_channel_Walk-In: -0.1864
visit_type_Consultation: 0.0157
visit_type_Follow-up: -0.0157

=== Reminder Sent Analysis ===
Scaled coefficient: -0.5807
Scaled odds ratio: 0.5595
Unscaled coefficient: -1.1731
Unscaled odds ratio: 0.3094

=== Conclusions ===
Why do the odds ratios differ?
The scaled coefficient represents the change in log-odds for a one-standard-deviation increase in reminder_sent.
The unscaled coefficient represents the change in log-odds when reminder_sent changes from 0 (No) to 1 (Yes).

Conclusion:
Holding other features constant, sending a reminder is associated with an odds ratio of 0.3094 for no-show.
An

#**Task 3.3 — Decision tree and overfitting**


In [5]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
dt_clf_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_clf_depth3.fit(X_train, y_train)

print("=== Decision Tree Classifier ===")
print(f"Training Accuracy: {dt_clf.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {dt_clf.score(X_test, y_test):.4f}")
print(f"Gap: {dt_clf.score(X_train, y_train) - dt_clf.score(X_test, y_test):.4f}")
print("\n=== Decision Tree Classifier (Depth 3) ===")
print(f"Training Accuracy: {dt_clf_depth3.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {dt_clf_depth3.score(X_test, y_test):.4f}")
print(f"Gap: {dt_clf_depth3.score(X_train, y_train) - dt_clf_depth3.score(X_test, y_test):.4f}")
print("\nWith depth3 there is less gap in between train and test accuracy which means the model is not overfitting.")

=== Decision Tree Classifier ===
Training Accuracy: 1.0000
Test Accuracy: 0.6717
Gap: 0.3283

=== Decision Tree Classifier (Depth 3) ===
Training Accuracy: 0.8220
Test Accuracy: 0.7778
Gap: 0.0442

With depth3 there is less gap in between train and test accuracy which means the model is not overfitting.


#**Task 3.4 — Random forest and a comparison table**


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

print("=== Random Forest Classifier ===")
print(f"Accuracy: {rf_clf.score(X_test, y_test):.4f}")
print(f"Recall: {recall_score(y_test, rf_clf.predict(X_test)):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, rf_clf.predict_proba(X_test)[:, 1]):.4f}")

# Combine feature names and importances into a DataFrame
feature_imp_df = pd.DataFrame({
    'Feature': rf_clf.feature_names_in_,
    'Importance': rf_clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_imp_df['Importance'] = feature_imp_df['Importance'].round(2)

# Print the table
print(feature_imp_df.to_string(index=False))

print(f"Top 3 Feature Importances: {rf_clf.feature_importances_.argsort()[-3:][::-1]}")

=== Random Forest Classifier ===
Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057
                Feature  Importance
              lead_days        0.22
                    age        0.20
            distance_km        0.19
                fee_inr        0.14
         prior_no_shows        0.11
          reminder_sent        0.05
  booking_channel_Phone        0.03
booking_channel_Walk-In        0.02
visit_type_Consultation        0.02
    booking_channel_App        0.02
   visit_type_Follow-up        0.02
Top 3 Feature Importances: [3 0 1]


In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# 1. Retrieve predictions from each model to build the exact comparison table
# Logistic Regression (Requires scaled features)
y_pred_lr = logreg_clf.predict(X_test_s)
y_prob_lr = logreg_clf.predict_proba(X_test_s)[:, 1]

# Decision Tree Depth 3 (Uses unscaled features)
y_pred_dt3 = dt_clf_depth3.predict(X_test)
y_prob_dt3 = dt_clf_depth3.predict_proba(X_test)[:, 1]

# Random Forest (Uses unscaled features)
y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

# 2. Build the comparison dictionary
metrics_data = {
    'Model': ['Logistic Regression', 'Decision Tree (Depth 3)', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt3),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt3),
        recall_score(y_test, y_pred_rf)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_dt3),
        roc_auc_score(y_test, y_prob_dt3)  # Use probability for AUC
    ]
}

# Overwrite ROC-AUC specifically with correct probabilities
metrics_data['ROC-AUC'][0] = roc_auc_score(y_test, y_prob_lr)
metrics_data['ROC-AUC'][1] = roc_auc_score(y_test, y_prob_dt3)
metrics_data['ROC-AUC'][2] = roc_auc_score(y_test, y_prob_rf)

comparison_df = pd.DataFrame(metrics_data)
display(comparison_df.round(4))

,Model,Accuracy,Recall,ROC-AUC
0,Logistic Regression,0.7879,0.3469,0.7889
1,Decision Tree (Depth 3),0.7778,0.2245,0.6997
2,Random Forest,0.7475,0.3061,0.7057


### **Which Model Looks Best and Why?**

- **Logistic Regression**: Accuracy = **78.79%**, Recall = **34.69%**, ROC-AUC = **78.89%**
- **Decision Tree (Depth 3)**: Accuracy = **77.78%**, Recall = **24.45%**, ROC-AUC = **69.97%**
- **Random Forest**: Accuracy = **74.75%**, Recall = **30.61%**, ROC-AUC = **70.57%**

**Conclusion:**
The **Logistic Regression** model currently shows the **best balance** for practical use because it achieves the highest Accuracy (78.79%) and the highest Recall (34.69%). In this clinic context, identifying actual no-shows is crucial (higher Recall), and Logistic Regression flags the most true no-shows while keeping overall errors minimal.

However, if we prioritize overall ranking ability and risk-scoring, the **Random Forest** is a very strong candidate due to having the highest overall discriminative power with an ROC-AUC of **70.57%**.

#**Task 3.5 — Tune, choose the model and the threshold, then test once**


In [8]:
# Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# (a) GridSearchCV - Tune Random Forest
rf = RandomForestClassifier(random_state=42)

# Define the hyperparameter grid to search
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8],
}

#  Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='roc_auc',
)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Inspect the best parameters
print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation Score:', grid_search.best_score_)


Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation Score: 0.7673242251739405


In [9]:
# (b) Comapare Four Models
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
forest_model = RandomForestClassifier(n_estimators=200, random_state=42)

tuned_rf = grid_search.best_estimator_

# Calculating cross-validation ROC-AUC
models = {
    'Logistic Regression': lr_pipe,
    'Decision Tree (Depth 3)': tree_model,
    'Random Forest': forest_model,
    'Tuned Random Forest': tuned_rf
}

print("=== Cross-Validated ROC-AUC ===")
print("=================================")
for model_name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{model_name} - Cross-Validated ROC-AUC: {cv_scores.mean():.4f}')


=== Cross-Validated ROC-AUC ===
Logistic Regression - Cross-Validated ROC-AUC: 0.7737
Decision Tree (Depth 3) - Cross-Validated ROC-AUC: 0.6784
Random Forest - Cross-Validated ROC-AUC: 0.7466
Tuned Random Forest - Cross-Validated ROC-AUC: 0.7673


In [11]:
# (c) Get out-of-fold predictions
best_model = lr_pipe
oof_probabilities = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method='predict_proba'
)[:, 1]
# Comapare thresholds 0.5 and 0.35
thresholds = [0.5, 0.35]

print("=== Comparison of Thresholds ===")
print("=================================")

for threshold in thresholds:
    y_pred = (oof_probabilities >= threshold).astype(int)
    print(f"Threshold: {threshold}")
    print(f"Precision: {precision_score(y_train, y_pred):.2f}")
    print(f"Recall: {recall_score(y_train, y_pred):.2f}")
    print(f"F1 Score: {f1_score(y_train, y_pred):.2f}")
    print(f"Number Flagged: {y_pred.sum()}\n")

=== Comparison of Thresholds ===
Threshold: 0.5
Precision: 0.58
Recall: 0.25
F1 Score: 0.35
Number Flagged: 73

Threshold: 0.35
Precision: 0.51
Recall: 0.47
F1 Score: 0.49
Number Flagged: 154



####**Final threshold = 0.35**

Why ?

Because recall increases from 0.25 to 0.47,and F1 improves from 0.35 to 0.49.

Higher recall is important because we want to identify more patients who might miss their appointments.

In [12]:
# (d) Train final model and evaluate on test data
best_model = lr_pipe
best_model.fit(X_train, y_train)
y_test_prob = best_model.predict_proba(X_test)[:,1]

final_threshold = 0.35
y_test_pred = (y_test_prob >= final_threshold).astype(int)

print("\n=== Final Model Evaluation ===")

print(f"\nFinal Threshold: {final_threshold}")
print(f"Precision: {precision_score(y_test, y_test_pred):.2f}")
print(f"Recall: {recall_score(y_test, y_test_pred):.2f}")
print(f"F1 Score: {f1_score(y_test, y_test_pred):.2f}")
print(f"Number Flagged: {y_test_pred.sum()}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_test_prob):.2f}")


=== Final Model Evaluation ===

Final Threshold: 0.35
Precision: 0.57
Recall: 0.57
F1 Score: 0.57
Number Flagged: 49
ROC-AUC: 0.79


In [14]:
# (e) Generate test_risk_scores.csv
test_risk_scores = pd.DataFrame({
    'appointment_id': original_clean_df.loc[X_test.index, "appointment_id"].values,
    'no_show': y_test.values,
    'risk_prob': y_test_prob.round(2),
    'flag': (y_test_prob >= final_threshold).astype(int)
})

test_risk_scores.to_csv('test_risk_scores.csv', index=False)
print("\n=== test_risk_scores ===\n")
print(test_risk_scores.head())
print("\nFile saved successfully !!")


=== test_risk_scores ===

  appointment_id  no_show  risk_prob  flag
0          A0217        0       0.02     0
1          A0336        0       0.15     0
2          A0508        0       0.52     1
3          A0314        0       0.19     0
4          A0089        0       0.17     0

File saved successfully !!


##**Final Conclusion :**

Logistic regression was selected because it achieved the highest cross-validation ROC-AUC of approximately 0.77. A threshold of 0.35 was selected because it provided a better F1 score and higher recall than the default threshold 0.5. On the test set, the model achieved approximately 0.57 precision, 0.57 recall and 0.79 ROC-AUC. The clinic can use these risk scores to identify patients who may need follow-up calls before their appointments.